# **Gradio Demo**
---
**DISCLAIMER**: Research prototype only. NOT for clinical use.

I wrote a standalone Gradio app to `app_or_demo/gradio_app.py` that accepts a chest X-ray image and returns a TB probability score.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import load_config
from src.paths import get_paths

cfg   = load_config()
paths = get_paths()

print('Notebook 17: Gradio Demo Writer')
print()
try:
    import gradio
    print(f'Gradio {gradio.__version__} is installed.')
except ImportError:
    print('Gradio not installed.')
    print('Install with:  pip install gradio')

Notebook 17: Gradio Demo Writer

Gradio not installed.
Install with:  pip install gradio


## Inference Pipeline — Step by Step

1. **Load image** — PIL converts any format to RGB (3-channel)
2. **Resize** — torchvision resizes to 224x224 (ResNet standard input)
3. **Normalise** — subtract ImageNet mean, divide by std
4. **ToTensor** — converts PIL to float32 torch Tensor
5. **Unsqueeze** — adds batch dimension: shape becomes (1, 3, 224, 224)
6. **Forward pass** — ResNet-18 produces raw logits of shape (1, 2)
7. **Softmax** — converts logits to probabilities summing to 1.0
8. **Return** — TB probability, interpretation text, and disclaimer

In [2]:
import torch
import numpy as np
from PIL import Image
from src.model import build_model
from src.data_utils import build_transforms

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = cfg['data']['image_size']

# Create a mock grayscale image (random noise) as a stand-in for a CXR
rng      = np.random.default_rng(42)
pixels   = rng.integers(0, 256, (256, 256), dtype=np.uint8)
mock_pil = Image.fromarray(pixels, mode='L')
print(f'Step 1 - Input image:  mode={mock_pil.mode}, size={mock_pil.size}')

# Apply val transform
val_transform = build_transforms(
    image_size=IMAGE_SIZE, split='val',
    normalize_mean=cfg['augmentation']['normalize_mean'],
    normalize_std=cfg['augmentation']['normalize_std'],
)
input_t = val_transform(mock_pil.convert('RGB'))
print(f'Steps 2-4 - After transform: shape={tuple(input_t.shape)}, '
      f'min={input_t.min():.3f}, max={input_t.max():.3f}')

# Add batch dimension
input_t = input_t.unsqueeze(0)
print(f'Step 5 - After unsqueeze: shape={tuple(input_t.shape)}')

# Forward pass
model = build_model(pretrained=False, num_classes=2, dropout=0.5).to(DEVICE)
model_path = paths['centralised_model_dir'] / 'best_model.pth'
if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    print(f'Step 6 - Loaded trained weights from {model_path.name}')
else:
    print('Step 6 - WARNING: No trained model found. Using random weights (output is meaningless).')
model.eval()
with torch.no_grad():
    logits = model(input_t.to(DEVICE))
    probs  = torch.softmax(logits, dim=1)
print(f'Step 7 - Logits: {logits.squeeze().cpu().numpy().round(4)}')
print(f'Step 7 - Probs:  {probs.squeeze().cpu().numpy().round(4)}')
print(f'         P(TB Negative) = {float(probs[0,0]):.4f}')
print(f'         P(TB Positive) = {float(probs[0,1]):.4f}')

Step 1 - Input image:  mode=L, size=(256, 256)
Steps 2-4 - After transform: shape=(3, 224, 224), min=-2.032, max=2.413
Step 5 - After unsqueeze: shape=(1, 3, 224, 224)
Step 6 - Loaded trained weights from best_model.pth
Step 7 - Logits: [ 0.2689 -0.8109]
Step 7 - Probs:  [0.7465 0.2535]
         P(TB Negative) = 0.7465
         P(TB Positive) = 0.2535


## Write the Gradio App Script

I now write the complete self-contained inference app to `app_or_demo/gradio_app.py`.

In [3]:
app_path = project_root / 'app_or_demo' / 'gradio_app.py'
app_path.parent.mkdir(parents=True, exist_ok=True)

lines = [
    '#!/usr/bin/env python3',
    '# app_or_demo/gradio_app.py',
    '# FedTB-Nigeria Gradio Inference Demo',
    '# DISCLAIMER: Research prototype. NOT for clinical use.',
    '# Usage: pip install gradio && python app_or_demo/gradio_app.py',
    '',
    'import sys',
    'from pathlib import Path',
    'sys.path.insert(0, str(Path(__file__).parent.parent))',
    '',
    'import torch',
    'import gradio as gr',
    '',
    'from src.config import load_config',
    'from src.paths import get_paths',
    'from src.model import build_model',
    'from src.data_utils import build_transforms',
    '',
    'cfg    = load_config()',
    'paths  = get_paths()',
    'DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")',
    '',
    'val_transform = build_transforms(',
    '    image_size=cfg["data"]["image_size"], split="val",',
    '    normalize_mean=cfg["augmentation"]["normalize_mean"],',
    '    normalize_std=cfg["augmentation"]["normalize_std"],',
    ')',
    '',
    'MODEL_LOADED = False',
    'model = build_model(pretrained=False, num_classes=2, dropout=0.5)',
    'model_path = paths["centralised_model_dir"] / "best_model.pth"',
    'if model_path.exists():',
    '    model.load_state_dict(torch.load(model_path, map_location=DEVICE))',
    '    model = model.to(DEVICE)',
    '    model.eval()',
    '    MODEL_LOADED = True',
    '    print(f"Model loaded from {model_path}")',
    'else:',
    '    print(f"WARNING: model not found at {model_path}. Run Notebook 07 first.")',
    '',
    'DISCLAIMER = (',
    '    "RESEARCH PROTOTYPE - NOT FOR CLINICAL USE.\\n"',
    '    "Do not use this output for diagnostic or treatment decisions.\\n"',
    '    "Always consult a qualified medical professional."',
    ')',
    '',
    'def predict_tb(image_pil):',
    '    if not MODEL_LOADED:',
    '        return {"Error": 1.0}, "Model not loaded. Run Notebook 07 first."',
    '    if image_pil is None:',
    '        return {"TB Negative": 0.5, "TB Positive": 0.5}, DISCLAIMER',
    '    img_rgb = image_pil.convert("RGB")',
    '    t = val_transform(img_rgb).unsqueeze(0).to(DEVICE)',
    '    import torch',
    '    with torch.no_grad():',
    '        probs = torch.softmax(model(t), dim=1)[0]',
    '    p_neg, p_pos = float(probs[0]), float(probs[1])',
    '    conf = "High" if max(p_neg, p_pos) > 0.80 else "Low"',
    '    interp = (',
    '        f"TB Probability: {p_pos:.1%}\\n"',
    '        f"Confidence: {conf}\\n\\n"',
    '        + DISCLAIMER',
    '    )',
    '    return {"TB Negative": p_neg, "TB Positive": p_pos}, interp',
    '',
    'demo = gr.Interface(',
    '    fn=predict_tb,',
    '    inputs=gr.Image(type="pil", label="Upload Chest X-Ray Image"),',
    '    outputs=[',
    '        gr.Label(num_top_classes=2, label="TB Classification Probability"),',
    '        gr.Textbox(label="Interpretation and Disclaimer", lines=8),',
    '    ],',
    '    title="FedTB-Nigeria: TB Detection Research Demo",',
    '    description="RESEARCH PROTOTYPE ONLY - NOT FOR CLINICAL USE.",',
    '    theme=gr.themes.Soft(),',
    '    allow_flagging="never",',
    ')',
    '',
    'if __name__ == "__main__":',
    '    demo.launch(share=False, server_port=7860)',
]

with open(app_path, 'w') as f:
    f.write('\n'.join(lines))

print(f'Gradio app written: {app_path}')
print(f'Lines: {len(lines)}')
print()
print('To run the demo:')
print('  pip install gradio')
print('  python app_or_demo/gradio_app.py')
print('  Open http://localhost:7860 in your browser')

Gradio app written: c:\Users\Peter\Documents\projects\Data_science\fedtb_nigeria\app_or_demo\gradio_app.py
Lines: 80

To run the demo:
  pip install gradio
  python app_or_demo/gradio_app.py
  Open http://localhost:7860 in your browser


In [4]:
# Verify syntax is valid Python
import ast
with open(app_path) as f:
    src = f.read()
try:
    ast.parse(src)
    print('gradio_app.py: Python syntax valid.')
except SyntaxError as e:
    print(f'SyntaxError: {e}')

gradio_app.py: Python syntax valid.


## DISCLAIMER

This Gradio demo is a **research prototype for portfolio demonstration only**.

- NOT clinically validated.
- MUST NOT be used to make diagnostic or treatment decisions.
- Any clinical deployment requires ethics approval, regulatory clearance, and prospective validation.
- Always consult a qualified medical professional for health concerns.

---